# Notebook 05 — Front Raw-Video Detection

**Objective:** run the accepted Front detector over one USER-selected raw video using the operational confidence threshold defined once in CONFIG. Only predictions returned at that threshold are valid detections for overlays, counts, CSV evidence, and later tracking input.

`DETECTION_CONF` is a prediction confidence threshold. `mAP50` is average precision evaluated at IoU=0.50. They are different concepts.

This notebook performs detection only. It does not train, track, assign identities, calculate behavior, modify the model/dataset, or perform Notebook 06.

## 1. Experiment metadata, available videos, and CONFIG

In [13]:
from pathlib import Path
from datetime import datetime, timezone
import hashlib, json, os, platform, subprocess, sys, time
import cv2
import numpy as np
import pandas as pd
import torch
import ultralytics
import yaml

STARTED_AT = datetime.now(timezone.utc).isoformat()
PROJECT_ROOT = Path.cwd().resolve()
if PROJECT_ROOT.name == 'notebooks': PROJECT_ROOT = PROJECT_ROOT.parent
CONDA_ENV = os.environ.get('CONDA_DEFAULT_ENV', '')
GIT_COMMIT = subprocess.run(['git', 'rev-parse', 'HEAD'], cwd=PROJECT_ROOT, capture_output=True, text=True, check=True).stdout.strip()
CUDA_AVAILABLE = torch.cuda.is_available()
GPU_NAME = torch.cuda.get_device_name(0) if CUDA_AVAILABLE else 'NOT_AVAILABLE'
RAW_FRONT_DIR = PROJECT_ROOT / 'data' / 'raw' / 'front'
VIDEO_EXTENSIONS = {'.mp4', '.avi', '.mov', '.mkv'}
AVAILABLE_VIDEO_PATHS = sorted(path for path in RAW_FRONT_DIR.glob('*') if path.is_file() and path.suffix.lower() in VIDEO_EXTENSIONS) if RAW_FRONT_DIR.is_dir() else []
print('Available raw Front videos:')
if AVAILABLE_VIDEO_PATHS:
    for path in AVAILABLE_VIDEO_PATHS: print(f'- {path.name} ({path.stat().st_size / (1024 ** 2):.2f} MB)')
else:
    print(f'- NONE in {RAW_FRONT_DIR.relative_to(PROJECT_ROOT)}')
MODEL_PATH = PROJECT_ROOT / 'runs' / 'front' / 'yolov8n_front_v1_baseline' / 'weights' / 'best.pt'
EXPECTED_MODEL_SHA256 = '750b0f8a1621f7214c8122467e5360ada69673ae4a8dc5bf3fd7b1e280287738'
VIDEO_FILENAME = '4.mp4'  # USER-selected file in data/raw/front/
VIDEO_PATH = RAW_FRONT_DIR / VIDEO_FILENAME
EXPECTED_FISH_COUNT = 1
# Operational detection confidence threshold.
# Change ONLY this value to test another threshold, e.g. 0.60 or 0.70.
DETECTION_CONF = 0.68
NMS_IOU = 0.70
IMGSZ = 640
DEVICE = 0
if not (0.0 < DETECTION_CONF <= 1.0):
    raise ValueError(f'DETECTION_CONF must be in (0, 1], got {DETECTION_CONF}')
if not (0.0 < NMS_IOU <= 1.0):
    raise ValueError(f'NMS_IOU must be in (0, 1], got {NMS_IOU}')
CONF_TAG = f'conf{int(round(DETECTION_CONF * 100)):03d}'
FISH_COUNT_TAG = f'n{EXPECTED_FISH_COUNT}'
RUN_TAG = f'{CONF_TAG}_{FISH_COUNT_TAG}'
EXPERIMENT_ID = f'FRONT_VIDEO_DET_{CONF_TAG.upper()}_{FISH_COUNT_TAG.upper()}_001'
PROGRESS_INTERVAL = 200
FLOAT_TOLERANCE = 1e-6
OUTPUT_DIR = PROJECT_ROOT / 'outputs' / 'front' / 'detection' / RUN_TAG
CONFIG = {'experiment_id': EXPERIMENT_ID, 'model_path': str(MODEL_PATH.relative_to(PROJECT_ROOT)), 'expected_model_sha256': EXPECTED_MODEL_SHA256, 'video_path': str(VIDEO_PATH.relative_to(PROJECT_ROOT)), 'expected_fish_count': EXPECTED_FISH_COUNT, 'experimental_fish_count': EXPECTED_FISH_COUNT, 'detection_conf': DETECTION_CONF, 'nms_iou': NMS_IOU, 'imgsz': IMGSZ, 'device': DEVICE, 'conf_tag': CONF_TAG, 'fish_count_tag': FISH_COUNT_TAG, 'run_tag': RUN_TAG, 'output_dir': str(OUTPUT_DIR.relative_to(PROJECT_ROOT)), 'progress_interval': PROGRESS_INTERVAL}
print(f'experiment_id: {EXPERIMENT_ID}')
print(f'datetime_utc: {STARTED_AT}')
print(f'PROJECT_ROOT: {PROJECT_ROOT}')
print(f'Python executable: {sys.executable}')
print(f'Conda environment: {CONDA_ENV}')
print(f'Python: {platform.python_version()}; Torch: {torch.__version__}; Ultralytics: {ultralytics.__version__}')
print(f'CUDA available: {CUDA_AVAILABLE}; CUDA runtime: {torch.version.cuda}; GPU: {GPU_NAME}')
print(f'Git commit: {GIT_COMMIT}')
print('CONFIG — FRONT VIDEO DETECTION')
print(f'experiment_id: {EXPERIMENT_ID}')
print(f'video: {VIDEO_PATH.relative_to(PROJECT_ROOT)}')
print(f'model: {MODEL_PATH.relative_to(PROJECT_ROOT)}')
print(f'DETECTION_CONF: {DETECTION_CONF}')
print(f'NMS_IOU: {NMS_IOU}')
print(f'IMGSZ: {IMGSZ}')
print(f'DEVICE: {DEVICE}')
print(f'CONF_TAG: {CONF_TAG}')
print(f'FISH_COUNT_TAG: {FISH_COUNT_TAG}')
print(f'output_dir: {OUTPUT_DIR.relative_to(PROJECT_ROOT)}')
print('IMPORTANT: DETECTION_CONF is a prediction confidence threshold; mAP50 is AP evaluated at IoU=0.50.')
if not AVAILABLE_VIDEO_PATHS: raise FileNotFoundError(f'MISSING_RAW_VIDEO: copy a raw Front video manually into {RAW_FRONT_DIR}; no automatic download is performed.')
if VIDEO_PATH not in AVAILABLE_VIDEO_PATHS: raise FileNotFoundError(f'USER_CONFIRM_REQUIRED: VIDEO_FILENAME={VIDEO_FILENAME!r} is not one of the listed files.')

Available raw Front videos:
- 4.mp4 (54.59 MB)
experiment_id: FRONT_VIDEO_DET_CONF068_N1_001
datetime_utc: 2026-08-17T08:57:01.499973+00:00
PROJECT_ROOT: /home/diy-hus/fish
Python executable: /home/diy-hus/miniconda3/envs/fish/bin/python
Conda environment: fish
Python: 3.11.15; Torch: 2.13.0+cu130; Ultralytics: 8.4.120
CUDA available: True; CUDA runtime: 13.0; GPU: NVIDIA GeForce RTX 3050
Git commit: 4cfac689c08d0f3f25bdee9cb8aac99d3202b9ee
CONFIG — FRONT VIDEO DETECTION
experiment_id: FRONT_VIDEO_DET_CONF068_N1_001
video: data/raw/front/4.mp4
model: runs/front/yolov8n_front_v1_baseline/weights/best.pt
DETECTION_CONF: 0.68
NMS_IOU: 0.7
IMGSZ: 640
DEVICE: 0
CONF_TAG: conf068
FISH_COUNT_TAG: n1
output_dir: outputs/front/detection/conf068_n1
IMPORTANT: DETECTION_CONF is a prediction confidence threshold; mAP50 is AP evaluated at IoU=0.50.


## 2. Mandatory model and video preflight

In [14]:
def sha256_file(path, chunk_size=1024 * 1024):
    digest = hashlib.sha256()
    with path.open('rb') as handle:
        for chunk in iter(lambda: handle.read(chunk_size), b''): digest.update(chunk)
    return digest.hexdigest()
assert CONDA_ENV == 'fish', f'FAIL preflight: expected Conda env fish, found {CONDA_ENV!r}'
assert CUDA_AVAILABLE, 'FAIL preflight: CUDA is required because DEVICE=0.'
assert MODEL_PATH.is_file(), f'FAIL preflight: missing model {MODEL_PATH}'
MODEL_SHA256 = sha256_file(MODEL_PATH)
assert MODEL_SHA256 == EXPECTED_MODEL_SHA256, f'FAIL preflight: model SHA-256 mismatch: {MODEL_SHA256}'
assert VIDEO_PATH.is_file(), f'FAIL preflight: selected video does not exist: {VIDEO_PATH}'
assert VIDEO_PATH.parent.resolve() == RAW_FRONT_DIR.resolve(), 'FAIL preflight: VIDEO_PATH must be a direct child of data/raw/front/.'
VIDEO_SHA256 = sha256_file(VIDEO_PATH)
capture = cv2.VideoCapture(str(VIDEO_PATH))
if not capture.isOpened(): raise RuntimeError(f'FAIL preflight: OpenCV cannot open {VIDEO_PATH}')
VIDEO_FPS = float(capture.get(cv2.CAP_PROP_FPS))
VIDEO_FRAME_COUNT = int(capture.get(cv2.CAP_PROP_FRAME_COUNT))
VIDEO_WIDTH = int(capture.get(cv2.CAP_PROP_FRAME_WIDTH))
VIDEO_HEIGHT = int(capture.get(cv2.CAP_PROP_FRAME_HEIGHT))
ok, probe_frame = capture.read(); capture.release()
assert ok and probe_frame is not None, 'FAIL preflight: first frame cannot be decoded.'
assert VIDEO_FPS > 0 and VIDEO_FRAME_COUNT > 0 and VIDEO_WIDTH > 0 and VIDEO_HEIGHT > 0, 'FAIL preflight: invalid video metadata.'
VIDEO_DURATION_SEC = VIDEO_FRAME_COUNT / VIDEO_FPS
VIDEO_SIZE_BYTES = VIDEO_PATH.stat().st_size
if OUTPUT_DIR.exists() and any(OUTPUT_DIR.iterdir()): raise RuntimeError(f'FAIL preflight: preserve existing output before rerun: {OUTPUT_DIR}')
print(f'Model: {MODEL_PATH.relative_to(PROJECT_ROOT)}')
print(f'Model SHA-256: {MODEL_SHA256}')
print(f'Video: {VIDEO_PATH.relative_to(PROJECT_ROOT)}; size={VIDEO_SIZE_BYTES} bytes; SHA-256={VIDEO_SHA256}')
print(f'FPS: {VIDEO_FPS:.6f}; frames: {VIDEO_FRAME_COUNT}; duration: {VIDEO_DURATION_SEC:.3f} sec; resolution: {VIDEO_WIDTH}x{VIDEO_HEIGHT}')
print(f'Expected fish count: {EXPECTED_FISH_COUNT}')
print('PREFLIGHT_RESULT: PASS')

Model: runs/front/yolov8n_front_v1_baseline/weights/best.pt
Model SHA-256: 750b0f8a1621f7214c8122467e5360ada69673ae4a8dc5bf3fd7b1e280287738
Video: data/raw/front/4.mp4; size=57241627 bytes; SHA-256=3f8587344beba8dcc6f835d1ed94aa8daadb06a369b5b96977ee902f035b5700
FPS: 28.668432; frames: 3431; duration: 119.679 sec; resolution: 1280x960
Expected fish count: 1
PREFLIGHT_RESULT: PASS


## 3. Full-video detection at the configured operational threshold

Only detections returned by `model.predict(conf=DETECTION_CONF)` are drawn, counted, saved, and made available to later notebooks. No second confidence filter is applied.

In [15]:
from ultralytics import YOLO
print(f'Loading detector: {MODEL_PATH.relative_to(PROJECT_ROOT)}')
MODEL_OBJECT = YOLO(str(MODEL_PATH), task='detect')
CLASS_NAMES = MODEL_OBJECT.names
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
OVERLAY_PATH = OUTPUT_DIR / 'front_detection_overlay.mp4'
FRAME_COUNTS_PATH = OUTPUT_DIR / 'frame_counts.csv'
DETECTIONS_PATH = OUTPUT_DIR / 'detections.csv'
video_capture = cv2.VideoCapture(str(VIDEO_PATH))
if not video_capture.isOpened(): raise RuntimeError('FAIL: cannot reopen selected video for inference.')
writer = cv2.VideoWriter(str(OVERLAY_PATH), cv2.VideoWriter_fourcc(*'mp4v'), VIDEO_FPS, (VIDEO_WIDTH, VIDEO_HEIGHT))
if not writer.isOpened(): video_capture.release(); raise RuntimeError(f'FAIL: cannot create overlay {OVERLAY_PATH}')
frame_rows, detection_rows = [], []
INFERENCE_START = time.perf_counter(); frame_index = 0
print(f'{RUN_TAG.upper()} — starting | confidence threshold={DETECTION_CONF:.2f} | expected fish={EXPECTED_FISH_COUNT} | NMS IoU={NMS_IOU:.2f} | imgsz={IMGSZ}')
try:
    while True:
        ok, frame = video_capture.read()
        if not ok: break
        result = MODEL_OBJECT.predict(source=frame, conf=DETECTION_CONF, iou=NMS_IOU, imgsz=IMGSZ, device=DEVICE, verbose=False)[0]
        boxes = result.boxes
        confidences = boxes.conf.detach().cpu().numpy().astype(float) if boxes is not None else np.array([], dtype=float)
        classes = boxes.cls.detach().cpu().numpy().astype(int) if boxes is not None else np.array([], dtype=int)
        xyxy = boxes.xyxy.detach().cpu().numpy().astype(float) if boxes is not None else np.empty((0, 4), dtype=float)
        if len(confidences) and float(confidences.min()) < DETECTION_CONF - FLOAT_TOLERANCE: raise RuntimeError('FAIL: framework returned a detection below DETECTION_CONF.')
        time_sec = frame_index / VIDEO_FPS
        frame_rows.append({'frame_index': frame_index, 'time_sec': time_sec, 'n_detections': int(len(confidences)), 'mean_conf': float(confidences.mean()) if len(confidences) else float('nan'), 'median_conf': float(np.median(confidences)) if len(confidences) else float('nan'), 'min_conf': float(confidences.min()) if len(confidences) else float('nan'), 'max_conf': float(confidences.max()) if len(confidences) else float('nan')})
        overlay = frame.copy()
        for class_id, confidence, coords in zip(classes, confidences, xyxy):
            x1, y1, x2, y2 = coords.tolist(); cx, cy = (x1 + x2) / 2, (y1 + y2) / 2
            detection_rows.append({'frame_index': frame_index, 'time_sec': time_sec, 'class_id': int(class_id), 'confidence': float(confidence), 'x1': x1, 'y1': y1, 'x2': x2, 'y2': y2, 'cx': cx, 'cy': cy})
            p1, p2 = (int(round(x1)), int(round(y1))), (int(round(x2)), int(round(y2)))
            cv2.rectangle(overlay, p1, p2, (0, 220, 0), 2)
            label = f'{CLASS_NAMES.get(int(class_id), class_id)} {confidence:.2f}'
            cv2.putText(overlay, label, (p1[0], max(18, p1[1] - 6)), cv2.FONT_HERSHEY_SIMPLEX, 0.55, (0, 220, 0), 2, cv2.LINE_AA)
        header = f'{RUN_TAG.upper()} | threshold={DETECTION_CONF:.2f} | frame {frame_index}/{VIDEO_FRAME_COUNT - 1} | t={time_sec:.2f}s | detections={len(confidences)}'
        cv2.rectangle(overlay, (0, 0), (min(VIDEO_WIDTH, 760), 34), (0, 0, 0), -1)
        cv2.putText(overlay, header, (10, 24), cv2.FONT_HERSHEY_SIMPLEX, 0.65, (255, 255, 255), 2, cv2.LINE_AA)
        writer.write(overlay); frame_index += 1
        if frame_index % PROGRESS_INTERVAL == 0 or frame_index == VIDEO_FRAME_COUNT:
            elapsed = time.perf_counter() - INFERENCE_START
            print(f'Processed {frame_index}/{VIDEO_FRAME_COUNT} | elapsed {elapsed:.1f}s | processing FPS {frame_index / elapsed:.2f}')
finally:
    video_capture.release(); writer.release()
PROCESSING_RUNTIME_SEC = time.perf_counter() - INFERENCE_START
if frame_index != VIDEO_FRAME_COUNT: raise RuntimeError(f'FAIL: processed {frame_index}/{VIDEO_FRAME_COUNT} frames; evidence is incomplete.')
FRAME_COUNTS_DF = pd.DataFrame(frame_rows)
DETECTIONS_DF = pd.DataFrame(detection_rows, columns=['frame_index', 'time_sec', 'class_id', 'confidence', 'x1', 'y1', 'x2', 'y2', 'cx', 'cy'])
FRAME_COUNTS_DF.to_csv(FRAME_COUNTS_PATH, index=False); DETECTIONS_DF.to_csv(DETECTIONS_PATH, index=False)
print(f'Created {OVERLAY_PATH.relative_to(PROJECT_ROOT)} ({OVERLAY_PATH.stat().st_size} bytes)')
print(f'Created {FRAME_COUNTS_PATH.relative_to(PROJECT_ROOT)} ({len(FRAME_COUNTS_DF)} records)')
print(f'Created {DETECTIONS_PATH.relative_to(PROJECT_ROOT)} ({len(DETECTIONS_DF)} records)')

Loading detector: runs/front/yolov8n_front_v1_baseline/weights/best.pt
CONF068_N1 — starting | confidence threshold=0.68 | expected fish=1 | NMS IoU=0.70 | imgsz=640
Processed 200/3431 | elapsed 4.6s | processing FPS 43.09
Processed 400/3431 | elapsed 8.7s | processing FPS 46.19
Processed 600/3431 | elapsed 12.8s | processing FPS 47.00
Processed 800/3431 | elapsed 16.9s | processing FPS 47.34
Processed 1000/3431 | elapsed 21.1s | processing FPS 47.38
Processed 1200/3431 | elapsed 24.9s | processing FPS 48.12
Processed 1400/3431 | elapsed 28.7s | processing FPS 48.81
Processed 1600/3431 | elapsed 32.4s | processing FPS 49.34
Processed 1800/3431 | elapsed 36.2s | processing FPS 49.69
Processed 2000/3431 | elapsed 40.1s | processing FPS 49.83
Processed 2200/3431 | elapsed 44.2s | processing FPS 49.73
Processed 2400/3431 | elapsed 48.6s | processing FPS 49.38
Processed 2600/3431 | elapsed 52.8s | processing FPS 49.24
Processed 2800/3431 | elapsed 56.9s | processing FPS 49.25
Processed 3000

## 4. Raw-video count and retained-confidence diagnostics

In [16]:
COUNTS = FRAME_COUNTS_DF['n_detections'].to_numpy(dtype=float)
COUNT_ERRORS = COUNTS - EXPECTED_FISH_COUNT
MEAN_DETECTIONS = float(COUNTS.mean()); MEDIAN_DETECTIONS = float(np.median(COUNTS))
MIN_DETECTIONS = int(COUNTS.min()); MAX_DETECTIONS = int(COUNTS.max())
EXACT_COUNT_FRAMES = int(np.sum(COUNT_ERRORS == 0)); EXACT_COUNT_RATE = float(np.mean(COUNT_ERRORS == 0))
UNDERCOUNT_FRAMES = int(np.sum(COUNT_ERRORS < 0)); UNDERCOUNT_RATE = float(np.mean(COUNT_ERRORS < 0))
OVERCOUNT_FRAMES = int(np.sum(COUNT_ERRORS > 0)); OVERCOUNT_RATE = float(np.mean(COUNT_ERRORS > 0))
COUNT_MAE = float(np.mean(np.abs(COUNT_ERRORS))); COUNT_RMSE = float(np.sqrt(np.mean(COUNT_ERRORS ** 2))); COUNT_BIAS = float(np.mean(COUNT_ERRORS))
ZERO_DETECTION_FRAMES = int(np.sum(COUNTS == 0)); PROCESSING_FPS = float(len(COUNTS) / PROCESSING_RUNTIME_SEC)
KEPT_CONFIDENCES = DETECTIONS_DF['confidence'].to_numpy(dtype=float)
if len(KEPT_CONFIDENCES):
    MEAN_CONFIDENCE = float(KEPT_CONFIDENCES.mean()); MEDIAN_CONFIDENCE = float(np.median(KEPT_CONFIDENCES))
    MIN_CONFIDENCE = float(KEPT_CONFIDENCES.min()); MAX_CONFIDENCE = float(KEPT_CONFIDENCES.max())
    Q25_CONFIDENCE = float(np.quantile(KEPT_CONFIDENCES, 1 / 4)); Q75_CONFIDENCE = float(np.quantile(KEPT_CONFIDENCES, 3 / 4))
    assert MIN_CONFIDENCE >= DETECTION_CONF - FLOAT_TOLERANCE, 'FAIL: retained confidence below operational threshold.'
else:
    MEAN_CONFIDENCE = MEDIAN_CONFIDENCE = MIN_CONFIDENCE = MAX_CONFIDENCE = Q25_CONFIDENCE = Q75_CONFIDENCE = None
COUNT_DIAGNOSTICS = {'detection_conf': DETECTION_CONF, 'frames': len(COUNTS), 'mean_detections': MEAN_DETECTIONS, 'median_detections': MEDIAN_DETECTIONS, 'min_detections': MIN_DETECTIONS, 'max_detections': MAX_DETECTIONS, 'exact_count_frames': EXACT_COUNT_FRAMES, 'exact_count_rate': EXACT_COUNT_RATE, 'undercount_frames': UNDERCOUNT_FRAMES, 'undercount_rate': UNDERCOUNT_RATE, 'overcount_frames': OVERCOUNT_FRAMES, 'overcount_rate': OVERCOUNT_RATE, 'count_MAE': COUNT_MAE, 'count_RMSE': COUNT_RMSE, 'count_bias': COUNT_BIAS, 'zero_detection_frames': ZERO_DETECTION_FRAMES, 'processing_FPS': PROCESSING_FPS}
CONFIDENCE_DIAGNOSTICS = {'detections_retained': len(KEPT_CONFIDENCES), 'mean_confidence': MEAN_CONFIDENCE, 'median_confidence': MEDIAN_CONFIDENCE, 'min_confidence': MIN_CONFIDENCE, 'max_confidence': MAX_CONFIDENCE, 'Q25_confidence': Q25_CONFIDENCE, 'Q75_confidence': Q75_CONFIDENCE}
print(f'RAW VIDEO COUNT DIAGNOSTICS AT CONF={DETECTION_CONF:.2f}')
print(COUNT_DIAGNOSTICS)
print('RETAINED DETECTION CONFIDENCE STATISTICS')
print(CONFIDENCE_DIAGNOSTICS)

RAW VIDEO COUNT DIAGNOSTICS AT CONF=0.68
{'detection_conf': 0.68, 'frames': 3431, 'mean_detections': 0.6706499562809677, 'median_detections': 1.0, 'min_detections': 0, 'max_detections': 1, 'exact_count_frames': 2301, 'exact_count_rate': 0.6706499562809677, 'undercount_frames': 1130, 'undercount_rate': 0.32935004371903237, 'overcount_frames': 0, 'overcount_rate': 0.0, 'count_MAE': 0.32935004371903237, 'count_RMSE': 0.573890271497115, 'count_bias': -0.32935004371903237, 'zero_detection_frames': 1130, 'processing_FPS': 49.28498286307921}
RETAINED DETECTION CONFIDENCE STATISTICS
{'detections_retained': 2301, 'mean_confidence': 0.7486294479486166, 'median_confidence': 0.7389723658561707, 'min_confidence': 0.6800433397293091, 'max_confidence': 0.8420991897583008, 'Q25_confidence': 0.717729926109314, 'Q75_confidence': 0.78428053855896}


## 5. Save compact evidence

In [17]:
LOG_DIR = PROJECT_ROOT / 'logs' / 'detection' / EXPERIMENT_ID
RESULTS_DIR = PROJECT_ROOT / 'results' / 'detection'
LOG_DIR.mkdir(parents=True, exist_ok=True); RESULTS_DIR.mkdir(parents=True, exist_ok=True)
CONFIG_PATH = LOG_DIR / 'config.yaml'; ENVIRONMENT_PATH = LOG_DIR / 'environment.txt'; SUMMARY_PATH = LOG_DIR / 'summary.json'
VIDEO_SUMMARY_PATH = RESULTS_DIR / f'front_video_detection_{RUN_TAG}_summary.csv'
COUNT_DISTRIBUTION_PATH = RESULTS_DIR / f'front_video_detection_{RUN_TAG}_count_distribution.csv'
CONFIG_EVIDENCE = dict(CONFIG, video_sha256=VIDEO_SHA256, video_fps=VIDEO_FPS, video_frame_count=VIDEO_FRAME_COUNT, video_duration_sec=VIDEO_DURATION_SEC, video_resolution=f'{VIDEO_WIDTH}x{VIDEO_HEIGHT}', git_commit=GIT_COMMIT)
CONFIG_PATH.write_text(yaml.safe_dump(CONFIG_EVIDENCE, sort_keys=False), encoding='utf-8')
ENVIRONMENT_LINES = [f'experiment_id={EXPERIMENT_ID}', f'datetime_utc={STARTED_AT}', f'git_commit={GIT_COMMIT}', f'python_executable={sys.executable}', f'python={platform.python_version()}', f'conda_env={CONDA_ENV}', f'torch={torch.__version__}', f'cuda_runtime={torch.version.cuda}', f'cuda_available={CUDA_AVAILABLE}', f'gpu={GPU_NAME}', f'ultralytics={ultralytics.__version__}', f'opencv={cv2.__version__}']
ENVIRONMENT_PATH.write_text('\n'.join(ENVIRONMENT_LINES) + '\n', encoding='utf-8')
SUMMARY_ROW = {**{'experiment_id': EXPERIMENT_ID, 'model_sha256': MODEL_SHA256, 'video': str(VIDEO_PATH.relative_to(PROJECT_ROOT)), 'video_sha256': VIDEO_SHA256, 'expected_fish_count': EXPECTED_FISH_COUNT, 'experimental_fish_count': EXPECTED_FISH_COUNT, 'nms_iou': NMS_IOU, 'imgsz': IMGSZ}, **COUNT_DIAGNOSTICS, **CONFIDENCE_DIAGNOSTICS}
pd.DataFrame([SUMMARY_ROW]).to_csv(VIDEO_SUMMARY_PATH, index=False)
distribution = FRAME_COUNTS_DF['n_detections'].value_counts().sort_index()
pd.DataFrame([{'detection_conf': DETECTION_CONF, 'count': int(count), 'frames': int(frames), 'frame_rate': float(frames / len(FRAME_COUNTS_DF))} for count, frames in distribution.items()]).to_csv(COUNT_DISTRIBUTION_PATH, index=False)
EVIDENCE_WARNINGS = []
EVIDENCE_WARNINGS.append('Previous N8 count diagnostics for this video were generated with an incorrect expected fish count and are superseded by the N1 run.')
if EXACT_COUNT_RATE < 1.0: EVIDENCE_WARNINGS.append(f'Exact-count rate at configured confidence {DETECTION_CONF:.2f} is below 1.0; model performance does not invalidate the experiment.')
if not len(KEPT_CONFIDENCES): EVIDENCE_WARNINGS.append('No detections met the operational confidence threshold; this is a performance warning, not an experiment failure.')
CHECKPOINT_RESULT = 'PASS_WITH_WARNING' if EVIDENCE_WARNINGS else 'PASS'
OUTPUT_FILES = [CONFIG_PATH, ENVIRONMENT_PATH, SUMMARY_PATH, VIDEO_SUMMARY_PATH, COUNT_DISTRIBUTION_PATH]
SUMMARY = {'experiment_id': EXPERIMENT_ID, 'model': str(MODEL_PATH.relative_to(PROJECT_ROOT)), 'model_sha256': MODEL_SHA256, 'video': str(VIDEO_PATH.relative_to(PROJECT_ROOT)), 'video_sha256': VIDEO_SHA256, 'video_size_bytes': VIDEO_SIZE_BYTES, 'expected_fish_count': EXPECTED_FISH_COUNT, 'experimental_fish_count': EXPECTED_FISH_COUNT, 'detection_conf': DETECTION_CONF, 'nms_iou': NMS_IOU, 'frames': VIDEO_FRAME_COUNT, 'fps': VIDEO_FPS, 'duration_sec': VIDEO_DURATION_SEC, 'resolution': f'{VIDEO_WIDTH}x{VIDEO_HEIGHT}', **COUNT_DIAGNOSTICS, **CONFIDENCE_DIAGNOSTICS, 'checkpoint_result': CHECKPOINT_RESULT, 'warnings': EVIDENCE_WARNINGS, 'heavy_output_files': [str(path.relative_to(PROJECT_ROOT)) for path in (OVERLAY_PATH, FRAME_COUNTS_PATH, DETECTIONS_PATH)], 'output_files': [str(path.relative_to(PROJECT_ROOT)) for path in OUTPUT_FILES], 'git_commit': GIT_COMMIT, 'next_step': 'Notebook 06 failure audit requires explicit USER approval.'}
SUMMARY_PATH.write_text(json.dumps(SUMMARY, indent=2, ensure_ascii=False), encoding='utf-8')
for path in OUTPUT_FILES: print(f'Created {path.relative_to(PROJECT_ROOT)} ({path.stat().st_size} bytes)')

Created logs/detection/FRONT_VIDEO_DET_CONF068_N1_001/config.yaml (715 bytes)
Created logs/detection/FRONT_VIDEO_DET_CONF068_N1_001/environment.txt (356 bytes)
Created logs/detection/FRONT_VIDEO_DET_CONF068_N1_001/summary.json (2391 bytes)
Created results/detection/front_video_detection_conf068_n1_summary.csv (959 bytes)
Created results/detection/front_video_detection_conf068_n1_count_distribution.csv (102 bytes)


## 6. Final Summary

In [18]:
FINAL_SUMMARY = {'experiment_id': EXPERIMENT_ID, 'model': str(MODEL_PATH.relative_to(PROJECT_ROOT)), 'model_sha256': MODEL_SHA256, 'video': str(VIDEO_PATH.relative_to(PROJECT_ROOT)), 'video_sha256': VIDEO_SHA256, 'expected_fish_count': EXPECTED_FISH_COUNT, 'experimental_fish_count': EXPECTED_FISH_COUNT, 'detection_conf': DETECTION_CONF, 'nms_iou': NMS_IOU, 'frames': VIDEO_FRAME_COUNT, 'fps': VIDEO_FPS, 'duration': VIDEO_DURATION_SEC, 'resolution': f'{VIDEO_WIDTH}x{VIDEO_HEIGHT}', 'mean_detections': MEAN_DETECTIONS, 'exact_count_rate': EXACT_COUNT_RATE, 'undercount_rate': UNDERCOUNT_RATE, 'overcount_rate': OVERCOUNT_RATE, 'count_MAE': COUNT_MAE, 'count_RMSE': COUNT_RMSE, 'count_bias': COUNT_BIAS, 'mean_confidence': MEAN_CONFIDENCE, 'median_confidence': MEDIAN_CONFIDENCE, 'min_confidence': MIN_CONFIDENCE, 'max_confidence': MAX_CONFIDENCE, 'processing_FPS': PROCESSING_FPS, 'checkpoint_result': CHECKPOINT_RESULT, 'warnings': EVIDENCE_WARNINGS, 'output_files': [str(path.relative_to(PROJECT_ROOT)) for path in OUTPUT_FILES], 'next_step': SUMMARY['next_step']}
print('FINAL SUMMARY')
for key, value in FINAL_SUMMARY.items(): print(f'{key}: {value}')

FINAL SUMMARY
experiment_id: FRONT_VIDEO_DET_CONF068_N1_001
model: runs/front/yolov8n_front_v1_baseline/weights/best.pt
model_sha256: 750b0f8a1621f7214c8122467e5360ada69673ae4a8dc5bf3fd7b1e280287738
video: data/raw/front/4.mp4
video_sha256: 3f8587344beba8dcc6f835d1ed94aa8daadb06a369b5b96977ee902f035b5700
expected_fish_count: 1
experimental_fish_count: 1
detection_conf: 0.68
nms_iou: 0.7
frames: 3431
fps: 28.66843196767745
duration: 119.67867666666668
resolution: 1280x960
mean_detections: 0.6706499562809677
exact_count_rate: 0.6706499562809677
undercount_rate: 0.32935004371903237
overcount_rate: 0.0
count_MAE: 0.32935004371903237
count_RMSE: 0.573890271497115
count_bias: -0.32935004371903237
mean_confidence: 0.7486294479486166
median_confidence: 0.7389723658561707
min_confidence: 0.6800433397293091
max_confidence: 0.8420991897583008
processing_FPS: 49.28498286307921
checkpoint_result: PASS_WITH_WARNING
warnings: ['Previous N8 count diagnostics for this video were generated with an incor